# Tutorial 8 — Vision-language models

[Lecture notes](../lectures/08_vlm.md) · [Slides](../slides/08_vlm.pptx) · [Exercise sheet](../exercises/08_vlm.md)

**40 minutes, CPU, NumPy only.** You will draw bar charts as pixel arrays, read them with a hand-written parser, and test whether a system uses its image by swapping the image for a blank, a counterfactual and a decoy. The "systems" are small rules that you can inspect, and none of them is a trained vision-language model. Nothing here measures a real model.

| Minutes | Step |
|---|---|
| 0–5 | Draw a chart and read it back |
| 5–10 | Blank and counterfactual images |
| 10–15 | Break the parser's layout assumption |
| 15–25 | Three systems, one question: does it use the image? |
| 25–40 | Your turn: the decoy test and a visual-token counter |

![Two charts with swapped heights and the same labels.](../data/chart-intervention.svg)

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Render controlled chart pixels

The image is 64 × 96 pixels with three bars: red is A, green is B, blue is C. One unit of height is 4 pixels. The parser knows where each bar sits and counts coloured pixels in one column per bar.

Before running: what should `read_heights` return for the values 3, 7, 5?

In [2]:
def chart(values):
    im=np.full((64,96,3),255,dtype=np.uint8)
    for i,h in enumerate(values):
        color=np.zeros(3,dtype=np.uint8); color[i]=200
        im[56-int(h)*4:56,8+28*i:24+28*i]=color
    return im

def read_heights(im):
    heights=[]
    for i in range(3):
        column=im[:56,12+28*i]
        colored=(column.min(axis=1)<100)&(column.max(axis=1)>100)
        heights.append(int(colored.sum())/4)
    return np.array(heights)
values=np.array([3,7,5]); original=chart(values)
print("image shape",original.shape,"read heights",read_heights(original))
assert np.array_equal(read_heights(original),values)

image shape (64, 96, 3) read heights [3. 7. 5.]


The parser returns 3, 7, 5 because it samples columns 12, 40 and 68, which is where `chart` drew the bars. That shared knowledge of the layout is the parser's whole skill. Section 3 removes it.

## 2. Intervene on evidence

Ask one question, "which bar is highest?", of three images: the original, a blank image and a counterfactual with the heights of A and B swapped.

Before running: write down the answer you want from a system that uses the image, for each of the three.

In [3]:
def answer(im):
    h=read_heights(im)
    if h.max()==0: return None
    if (h==h.max()).sum()>1: return "tie"
    return "ABC"[int(h.argmax())]
blank=np.full_like(original,255); swapped=chart([7,3,5])
for name,im in [("original",original),("blank",blank),("counterfactual",swapped)]:
    print(name,"highest bar",answer(im))
assert answer(original)=="B" and answer(swapped)=="A" and answer(blank) is None

original highest bar B
blank highest bar None
counterfactual highest bar A


The answer moves from B to A when the evidence moves, and the parser returns `None` on the blank image. A system that still says B on the blank image is answering from somewhere other than the pixels. Section 4 builds one.

## 3. Break the rendering assumptions

Shift the chart 20 pixels to the right. A person reads it the same way.

Before running: will the parser still answer B?

In [4]:
shifted=np.roll(original,20,axis=1)
print("shifted heights",read_heights(shifted),"answer",answer(shifted))
print("raw patch counts at patch size 14",(336//14)**2,(672//14)*(336//14))
assert (448//14)**2==1024
# Inspect pixel evidence for the highest bar: colored vertical runs.
print("green-bar center rows",np.where(original[:56,40,1]==200)[0])

shifted heights [0. 3. 7.] answer C
raw patch counts at patch size 14 576 1152
green-bar center rows [28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51
 52 53 54 55]


The parser reads heights 0, 3, 7 and answers C. Its three sampling columns now fall on white space, on the shifted red bar and on the shifted green bar, so every bar is credited to the wrong letter. The answer is confident and wrong, with no warning. The patch counts on the second line are the lecture's arithmetic: 336/14 = 24 and 24 × 24 = 576.

## 4. Does the system use the image?

Build 300 chart questions in which B is the tallest bar half the time, as if the dataset had a favourite answer. Three systems answer them:

- `prior`: never looks, always says B
- `weak`: reads the chart on about 15% of calls and otherwise guesses A, B or C with odds 1 : 2 : 1
- `parser`: always reads the chart

Each item also gets a **decoy**: the same question with another item's chart. The decoy answer is still scored against the real chart's label. Task 12 subtask 1 uses the same design with paintings.

Before running: predict each system's accuracy with the real image and with the decoy.

In [5]:
n=300
truth=rng.choice(3,size=n,p=[.25,.5,.25])
def heights_for(top):
    h=rng.permutation(np.arange(1,9))[:3]; h[[top,int(h.argmax())]]=h[[int(h.argmax()),top]]; return h
charts=[chart(heights_for(t)) for t in truth]
decoy_of=rng.permutation(n)                         # item i is shown the chart of item decoy_of[i]
def weak(im):
    if answer(im) is None: return "ABC"[rng.choice(3,p=[.25,.5,.25])]     # it never abstains
    return answer(im) if rng.random()<.15 else "ABC"[rng.choice(3,p=[.25,.5,.25])]
systems={"prior":lambda im:"B","weak":weak,"parser":answer}
gold=np.array(list("ABC"))[truth]
results={}
print("system   real image  decoy image  blank image: abstains")
for name,f in systems.items():
    real=np.array([f(charts[i]) for i in range(n)])                # predicted labels with the real image
    decoy=np.array([f(charts[decoy_of[i]]) for i in range(n)])     # and with the decoy
    abstain=np.mean([f(blank) is None for i in range(n)])
    results[name]=(real,decoy)
    print(f"{name:7s}  {np.mean(real==gold):10.3f}  {np.mean(decoy==gold):11.3f}  {abstain:21.3f}")
assert np.mean(results["parser"][0]==gold)==1.0 and np.array_equal(*results["prior"])

system   real image  decoy image  blank image: abstains
prior         0.467        0.467                  0.000
weak          0.447        0.387                  0.000
parser        1.000        0.333                  1.000


`prior` scores the same with either image, and its accuracy equals the share of items whose answer is B. A leaderboard that showed only the first column would rank it above chance and tell you nothing about vision. `weak` scores 0.447 with the real image and 0.387 with the decoy, and sits below `prior` on plain accuracy even though it is the one that looks. Is a gap of 0.06 on 300 items more than chance would give? Part A of your turn answers with a test.

The last column is abstention on a blank image. `prior` answers B to an image with no bars, which is the toy version of a hallucination.

## 5. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. The decoy test.** You get two boolean arrays over the same items: correct with the real image, and correct with the decoy. Count the discordant pairs: `b` items are right with the real image and wrong with the decoy, and `c` items are the reverse. If the image did not matter, each discordant pair would fall either way with probability 1/2. The one-sided exact McNemar p-value is the chance of `b` or more out of `b + c` fair coin flips:

$$p=\sum_{k=b}^{b+c}\binom{b+c}{k}\Big/2^{\,b+c}$$

Return `(b, c, p)`. With no discordant pairs, return `p = 1.0`.

In [ ]:
# YOUR TURN
def decoy_test(real_correct, decoy_correct):
    """Return (b, c, one-sided exact McNemar p-value). Use math.comb."""
    raise NotImplementedError("your turn")

In [7]:
# The lecture's table: 100 items, 30 right both times, 12 right only with the real image, 4 right only with the decoy, 54 wrong both times.
real_demo=np.array([1]*30+[1]*12+[0]*4+[0]*54,dtype=bool); decoy_demo=np.array([1]*30+[0]*12+[1]*4+[0]*54,dtype=bool)
b,c,p=decoy_test(real_demo,decoy_demo)
assert (b,c)==(12,4) and np.isclose(p,2517/65536)
assert np.isclose(decoy_test(np.array([1,1,1,1,1,1,1,0],dtype=bool),np.array([0,0,0,0,0,0,0,1],dtype=bool))[2],9/256)
assert decoy_test(real_demo,real_demo)==(0,0,1.0)
print("lecture table: b =",b," c =",c," p =",round(p,4))
for name,(real,decoy) in results.items():
    b,c,p=decoy_test(real==gold,decoy==gold)
    print(f"{name:7s} b = {b:3d}  c = {c:3d}  p = {p:.2g}  same label on real and decoy: {np.mean(real==decoy):.3f}")
print("A passed")

lecture table: b = 12  c = 4  p = 0.0384
prior   b =   0  c =   0  p = 1  same label on real and decoy: 1.000
weak    b =  76  c =  58  p = 0.071  same label on real and decoy: 0.393
parser  b = 200  c =   0  p = 6.2e-61  same label on real and decoy: 0.333
A passed


Read the three rows against the Task 12 rule: a multimodal run needs p < 0.05, and a text-only run must give real and decoy rows the same label at least 99% of the time. `prior` qualifies as a text-only run and `parser` as a multimodal run. `weak` fails both: p = 0.071, and its labels agree on 39% of rows. It does use the image, on about 15% of calls, and 300 items are too few to show it. How many items would you need?

**B. Count the visual tokens.** A vision transformer cuts an image of `height × width` pixels into square patches of side `patch` and keeps the whole patches. Some models then pool `pool × pool` neighbouring patches into one token. Return the number of tokens the decoder receives.

In [ ]:
# YOUR TURN
def visual_tokens(height, width, patch, pool=1):
    """Number of visual tokens: whole patches per side, pooled by `pool` per side."""
    raise NotImplementedError("your turn")

In [9]:
assert visual_tokens(224,224,32)==49 and visual_tokens(224,224,16)==196
assert visual_tokens(336,336,14)==576 and visual_tokens(448,448,14)==1024
assert visual_tokens(336,672,14)==1152                 # twice the width, twice the tokens
assert visual_tokens(896,896,14,pool=4)==256           # 64 x 64 patches pooled 4 x 4
assert visual_tokens(230,230,16)==196                  # a partial patch is dropped
for side in [224,448,896]:
    print(f"{side} x {side}, patch 14: {visual_tokens(side,side,14):5d} tokens")
print("B passed")

224 x 224, patch 14:   256 tokens
448 x 448, patch 14:  1024 tokens
896 x 896, patch 14:  4096 tokens
B passed


Doubling the side multiplies the tokens by four. Decoder attention cost grows with the square of the sequence length, so a document page at high resolution can cost more than the whole text prompt. Real processors also crop, tile and add special tokens: print the processor's output shapes for your model and count.

## Discussion

1. What does a blank-image baseline tell you that accuracy does not?
2. Can a correct numerical answer be visually ungrounded? Give an example from section 4.
3. Why does the hand-written parser beat a small VLM on this chart, and what breaks that advantage?

## From the toy to your task

- **Task 12, subtask 1.** Run the decoy test on your development set before the organisers run it on your test submission: predict once with each painting and once with a different painting under the same caption.
- **Task 6.** Blank the image for 50 development questions. The share you still answer correctly is what the question text gives away.
- **A recorded real error.** [One pretrained SmolVLM run](../validation/vlm_original_run.json) on [this chart](../validation/vlm_original_chart.png) answered A when the tallest bar was B. One example shows a failure and estimates nothing. `course/extensions/hf_vision.py --mode vlm` repeats it; see the [extension guide](../extensions/README.md).

Submit this notebook with both tests passing and a 150–250-word note: which toy system uses its image, the numbers that show it, and one way a real VLM's failure would look different.